In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import os

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

def generate_fraud_dataset(n_records=200000):
    """
    Generate synthetic financial transaction dataset
    with realistic fraud patterns
    
    n_records: Number of transaction records (1L-3L)
    """
    
    print(f"🔄 Generating {n_records:,} transaction records...")
    
    # Date range
    start_date = datetime(2023, 1, 1)
    end_date = datetime(2024, 12, 31)
    date_range = (end_date - start_date).days
    
    data = {
        'transaction_id': [f'TXN_{i:07d}' for i in range(1, n_records + 1)],
        'timestamp': [start_date + timedelta(days=random.randint(0, date_range), 
                                            hours=random.randint(0, 23),
                                            minutes=random.randint(0, 59)) 
                     for _ in range(n_records)],
        'customer_id': [f'CUST_{random.randint(1, 50000):06d}' for _ in range(n_records)],
        'merchant_id': [f'MERCH_{random.randint(1, 10000):05d}' for _ in range(n_records)],
        'transaction_amount': np.random.exponential(200, n_records),
        'merchant_category': np.random.choice([
            'Grocery', 'Restaurant', 'Gas Station', 'Online Retail',
            'Hotel', 'Airlines', 'Healthcare', 'Entertainment',
            'Utilities', 'Pharmacy', 'Electronics', 'Clothing'
        ], n_records),
        'card_type': np.random.choice(['Credit', 'Debit'], n_records),
        'transaction_type': np.random.choice(['Online', 'Offline', 'Mobile'], n_records),
        'country': np.random.choice([
            'USA', 'UK', 'Canada', 'India', 'Germany',
            'France', 'Australia', 'Mexico', 'Brazil', 'Nigeria'
        ], n_records),
        'device_type': np.random.choice(['Mobile', 'Desktop', 'Tablet', 'ATM', 'POS'], n_records),
    }
    
    df = pd.DataFrame(data)
    
    # Create fraud column with realistic distribution
    fraud_rate = 0.04  # 4% fraud rate
    n_frauds = int(n_records * fraud_rate)
    
    fraud_labels = np.array([1] * n_frauds + [0] * (n_records - n_frauds))
    np.random.shuffle(fraud_labels)
    df['is_fraud'] = fraud_labels
    
    # Add fraud-specific patterns
    fraud_indices = df[df['is_fraud'] == 1].index
    
    # Frauds tend to have higher amounts
    df.loc[fraud_indices, 'transaction_amount'] = np.random.exponential(500, len(fraud_indices))
    
    # Frauds tend to happen at different hours
    df.loc[fraud_indices, 'timestamp'] = df.loc[fraud_indices, 'timestamp'].apply(
        lambda x: x.replace(hour=random.randint(0, 23))
    )
    
    # Frauds often involve specific countries
    df.loc[fraud_indices, 'country'] = np.random.choice(['Nigeria', 'Russia', 'China'], len(fraud_indices))
    
    # Additional features
    df['transaction_hour'] = df['timestamp'].dt.hour
    df['transaction_day'] = df['timestamp'].dt.day
    df['transaction_month'] = df['timestamp'].dt.month
    df['transaction_day_of_week'] = df['timestamp'].dt.dayofweek
    
    # Introduce some missing values (realistic)
    missing_indices = np.random.choice(n_records, int(n_records * 0.02), replace=False)
    df.loc[missing_indices[:n_records//1000], 'transaction_hour'] = np.nan
    
    print(f"✅ Dataset generated successfully!")
    print(f"   - Total Records: {len(df):,}")
    print(f"   - Fraud Cases: {df['is_fraud'].sum():,} ({df['is_fraud'].mean()*100:.2f}%)")
    print(f"   - Genuine Cases: {(1-df['is_fraud']).sum():,}")
    
    return df

# Generate dataset
df_raw = generate_fraud_dataset(n_records=250000)  # 2.5L records

# Save dataset
os.makedirs('data', exist_ok=True)
df_raw.to_csv('data/raw_transactions.csv', index=False)
print("\n📁 Dataset saved to data/raw_transactions.csv")

# Display info
print("\n📊 Dataset Info:")
print(df_raw.head(10))
print("\n" + "="*50)
print(df_raw.info())
print("\n" + "="*50)
print(df_raw.describe())

🔄 Generating 250,000 transaction records...
✅ Dataset generated successfully!
   - Total Records: 250,000
   - Fraud Cases: 10,000 (4.00%)
   - Genuine Cases: 240,000

📁 Dataset saved to data/raw_transactions.csv

📊 Dataset Info:
  transaction_id           timestamp  customer_id  merchant_id  \
0    TXN_0000001 2024-10-16 03:01:00  CUST_010607  MERCH_09995   
1    TXN_0000002 2023-10-09 07:14:00  CUST_044501  MERCH_06467   
2    TXN_0000003 2023-05-23 23:06:00  CUST_027157  MERCH_05502   
3    TXN_0000004 2024-11-23 23:57:00  CUST_004958  MERCH_02380   
4    TXN_0000005 2024-07-12 02:37:00  CUST_048904  MERCH_05684   
5    TXN_0000006 2024-03-08 01:01:00  CUST_025810  MERCH_01762   
6    TXN_0000007 2023-04-06 06:14:00  CUST_003574  MERCH_02939   
7    TXN_0000008 2024-06-01 19:01:00  CUST_017071  MERCH_04160   
8    TXN_0000009 2024-07-28 06:45:00  CUST_022910  MERCH_07203   
9    TXN_0000010 2024-10-27 22:34:00  CUST_031769  MERCH_08466   

   transaction_amount merchant_category car

In [2]:
print(df_raw.shape)

print("\nMissing Values:")
print(df_raw.isnull().sum())

print("\nDuplicates:")
print(df_raw.duplicated().sum())

print("\nFraud Distribution:")
print(df_raw['is_fraud'].value_counts())

print("\nFraud Percentage:")
print(df_raw['is_fraud'].value_counts(normalize=True)*100)

(250000, 15)

Missing Values:
transaction_id               0
timestamp                    0
customer_id                  0
merchant_id                  0
transaction_amount           0
merchant_category            0
card_type                    0
transaction_type             0
country                      0
device_type                  0
is_fraud                     0
transaction_hour           250
transaction_day              0
transaction_month            0
transaction_day_of_week      0
dtype: int64

Duplicates:
0

Fraud Distribution:
is_fraud
0    240000
1     10000
Name: count, dtype: int64

Fraud Percentage:
is_fraud
0    96.0
1     4.0
Name: proportion, dtype: float64


In [3]:
import os

folders = [
    "data/raw",
    "data/processed",
    "notebooks",
    "src",
    "models",
    "reports",
    "dashboard",
    "images"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("✅ Project folders created successfully")

✅ Project folders created successfully


In [4]:
files = [
    "src/data_cleaner.py",
    "src/eda.py",
    "src/feature_engineering.py",
    "src/model_trainer.py",
    "src/model_evaluator.py",
    "dashboard/app.py",
    "README.md",
    "requirements.txt"
]

for file in files:
    open(file, "a").close()

print("✅ Project files created successfully")

✅ Project files created successfully


In [5]:
for root, dirs, files in os.walk("."):
    level = root.replace(".", "").count(os.sep)
    indent = " " * 4 * level
    print(f"{indent}{os.path.basename(root)}/")

    subindent = " " * 4 * (level + 1)
    for file in files:
        print(f"{subindent}{file}")

./
    day_01.ipynb
    FraudShield AI.docx
    README.md
    requirements.txt
    .ipynb_checkpoints/
        day_01-checkpoint.ipynb
    dashboard/
        app.py
    data/
        raw_transactions.csv
        processed/
        raw/
    images/
    models/
    notebooks/
    reports/
    src/
        data_cleaner.py
        eda.py
        feature_engineering.py
        model_evaluator.py
        model_trainer.py


In [10]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

class DataCleaner:
  """
FraudShield AI
Data Cleaning Module

Responsibilities:
- Missing value handling
- Duplicate removal
- Outlier treatment
- Data validation
"""

def __init__(self, df):
    self.df = df.copy()
    self.report = {}

# --------------------------------------------------
# Missing Values
# --------------------------------------------------
def handle_missing_values(self):
    """
    Handle missing values using:
    - Median for numeric columns
    - Mode for categorical columns
    """

    print("🔍 Handling missing values...")

    initial_missing = self.df.isnull().sum()
    self.report["missing_before"] = initial_missing.sum()

    # Numeric columns
    numeric_cols = self.df.select_dtypes(
        include=[np.number]
    ).columns

    for col in numeric_cols:
        if self.df[col].isnull().sum() > 0:
            self.df[col] = self.df[col].fillna(
                self.df[col].median()
            )

    # Categorical columns
    categorical_cols = self.df.select_dtypes(
        include=["object"]
    ).columns

    for col in categorical_cols:
        if self.df[col].isnull().sum() > 0:
            self.df[col] = self.df[col].fillna(
                self.df[col].mode()[0]
            )

    final_missing = self.df.isnull().sum()

    self.report["missing_after"] = final_missing.sum()

    print(
        f"✅ Missing values handled: "
        f"{self.report['missing_before']} → "
        f"{self.report['missing_after']}"
    )

    return self

# --------------------------------------------------
# Duplicate Removal
# --------------------------------------------------
def remove_duplicates(self):
    """
    Remove duplicate transaction IDs
    """

    print("🔍 Removing duplicates...")

    before_rows = len(self.df)

    self.df.drop_duplicates(
        subset=["transaction_id"],
        inplace=True
    )

    after_rows = len(self.df)

    duplicates_removed = before_rows - after_rows

    self.report["duplicates_removed"] = duplicates_removed

    print(
        f"✅ Duplicates removed: "
        f"{duplicates_removed}"
    )

    return self

# --------------------------------------------------
# Outlier Handling
# --------------------------------------------------
def handle_outliers(
    self,
    column="transaction_amount",
    threshold=1.5
):
    """
    Handle outliers using IQR capping
    """

    print(f"🔍 Handling outliers in {column}...")

    Q1 = self.df[column].quantile(0.25)
    Q3 = self.df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - threshold * IQR
    upper_bound = Q3 + threshold * IQR

    # Count outliers BEFORE clipping
    outlier_mask = (
        (self.df[column] < lower_bound)
        |
        (self.df[column] > upper_bound)
    )

    outlier_count = outlier_mask.sum()

    # Cap outliers
    self.df[column] = self.df[column].clip(
        lower=lower_bound,
        upper=upper_bound
    )

    self.report["outliers_handled"] = int(outlier_count)

    print(
        f"✅ Outliers capped: "
        f"{outlier_count}"
    )

    return self

# --------------------------------------------------
# Validation
# --------------------------------------------------
def validate_data(self):
    """
    Final validation check
    """

    print("\n" + "=" * 60)
    print("📋 VALIDATION REPORT")
    print("=" * 60)

    print(f"Dataset Shape: {self.df.shape}")

    print("\nMissing Values:")
    print(self.df.isnull().sum())

    print("\nDuplicate Records:")
    print(self.df.duplicated().sum())

    return self

# --------------------------------------------------
# Data Quality Report
# --------------------------------------------------
def generate_report(self):

    print("\n" + "=" * 60)
    print("📊 DATA QUALITY REPORT")
    print("=" * 60)

    print(f"Total Records: {len(self.df):,}")

    print(
        f"Missing Values Before: "
        f"{self.report.get('missing_before', 0)}"
    )

    print(
        f"Missing Values After : "
        f"{self.report.get('missing_after', 0)}"
    )

    print(
        f"Duplicates Removed   : "
        f"{self.report.get('duplicates_removed', 0)}"
    )

    print(
        f"Outliers Handled     : "
        f"{self.report.get('outliers_handled', 0)}"
    )

    print(
        f"Fraud Cases          : "
        f"{self.df['is_fraud'].sum():,}"
    )

    print(
        f"Fraud Percentage     : "
        f"{self.df['is_fraud'].mean()*100:.2f}%"
    )

    print("=" * 60)

    return self.report

# --------------------------------------------------
# Return Clean Dataset
# --------------------------------------------------
def get_cleaned_data(self):
    return self.df

In [9]:
from src.data_cleaner import DataCleaner

cleaner = DataCleaner(df_raw)

cleaner.handle_missing_values() \
       .remove_duplicates() \
       .handle_outliers() \
       .validate_data() \
       .generate_report()

df_clean = cleaner.get_cleaned_data()

ImportError: cannot import name 'DataCleaner' from 'src.data_cleaner' (C:\Users\Dell\FraudShield AI\src\data_cleaner.py)

In [11]:
print(DataCleaner)

<class '__main__.DataCleaner'>


In [12]:
dir(DataCleaner)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__']

In [13]:
help(DataCleaner)


Help on class DataCleaner in module __main__:

class DataCleaner(builtins.object)
 |  FraudShield AI
 |  Data Cleaning Module
 |
 |  Responsibilities:
 |  - Missing value handling
 |  - Duplicate removal
 |  - Outlier treatment
 |  - Data validation
 |
 |  Data descriptors defined here:
 |
 |  __dict__
 |      dictionary for instance variables
 |
 |  __weakref__
 |      list of weak references to the object



In [16]:
import importlib
import src.data_cleaner

importlib.reload(src.data_cleaner)

from src.data_cleaner import DataCleaner

In [17]:
print(dir(DataCleaner))

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', 'get_cleaned_data', 'handle_missing_values']


In [18]:
cleaner = DataCleaner(df_raw)

cleaner.handle_missing_values()

Handling missing values...
Done


In [19]:
df_clean = cleaner.get_cleaned_data()

df_clean.to_csv(
    "data/processed/clean_transactions.csv",
    index=False
)

print("✅ Clean dataset saved")

✅ Clean dataset saved


In [21]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings

warnings.filterwarnings('ignore')


class DataCleaner:
    """
    Comprehensive Data Cleaning & Preparation Module
    """

    def __init__(self, df):
        self.df = df.copy()
        self.report = {}
        self.encoders = {}

    # =====================================================
    # Missing Value Handling
    # =====================================================

    def handle_missing_values(self, strategy='median'):
        """
        Handle missing values

        Parameters:
        strategy : mean | median
        """

        print("🔍 Handling missing values...")

        initial_missing = self.df.isnull().sum()
        self.report['missing_values_before'] = initial_missing

        # Numeric Columns
        numeric_cols = self.df.select_dtypes(
            include=[np.number]
        ).columns

        for col in numeric_cols:

            if self.df[col].isnull().sum() > 0:

                if strategy == 'mean':
                    self.df[col].fillna(
                        self.df[col].mean(),
                        inplace=True
                    )

                else:
                    self.df[col].fillna(
                        self.df[col].median(),
                        inplace=True
                    )

        # Categorical Columns
        categorical_cols = self.df.select_dtypes(
            include=['object']
        ).columns

        for col in categorical_cols:

            if self.df[col].isnull().sum() > 0:

                self.df[col].fillna(
                    self.df[col].mode()[0],
                    inplace=True
                )

        final_missing = self.df.isnull().sum()

        self.report['missing_values_after'] = final_missing

        print("✅ Missing values handled")
        print(
            f"   - Missing values: "
            f"{initial_missing.sum()} → {final_missing.sum()}"
        )

        return self

    # =====================================================
    # Duplicate Removal
    # =====================================================

    def remove_duplicates(self):
        """
        Remove duplicate transaction records
        """

        print("🔍 Removing duplicates...")

        initial_rows = len(self.df)

        self.df.drop_duplicates(
            subset=['transaction_id'],
            inplace=True
        )

        final_rows = len(self.df)

        duplicates_removed = initial_rows - final_rows

        self.report['duplicates_removed'] = duplicates_removed

        print(
            f"✅ Duplicates removed: "
            f"{duplicates_removed} records"
        )

        return self

    # =====================================================
    # Outlier Handling
    # =====================================================

    def handle_outliers(
        self,
        method='iqr',
        threshold=1.5
    ):
        """
        Handle outliers using IQR capping
        """

        print("🔍 Handling outliers...")

        numeric_cols = ['transaction_amount']

        outliers_count = 0

        for col in numeric_cols:

            Q1 = self.df[col].quantile(0.25)
            Q3 = self.df[col].quantile(0.75)

            IQR = Q3 - Q1

            lower_bound = Q1 - threshold * IQR
            upper_bound = Q3 + threshold * IQR

            # Count BEFORE clipping
            outlier_mask = (
                (self.df[col] < lower_bound)
                |
                (self.df[col] > upper_bound)
            )

            outliers_count += outlier_mask.sum()

            # Cap values
            self.df[col] = self.df[col].clip(
                lower=lower_bound,
                upper=upper_bound
            )

        self.report['outliers_handled'] = int(outliers_count)

        print(
            f"✅ Outliers handled: "
            f"{outliers_count} values capped"
        )

        return self

    # =====================================================
    # Categorical Encoding
    # =====================================================

    def encode_categorical(self):
        """
        Label Encode categorical variables
        """

        print("🔍 Encoding categorical variables...")

        categorical_cols = self.df.select_dtypes(
            include=['object']
        ).columns

        categorical_cols = [
            col for col in categorical_cols
            if col not in [
                'transaction_id',
                'customer_id',
                'merchant_id'
            ]
        ]

        for col in categorical_cols:

            encoder = LabelEncoder()

            self.df[col + '_encoded'] = encoder.fit_transform(
                self.df[col]
            )

            self.encoders[col] = encoder

        print(
            f"✅ Encoded {len(categorical_cols)} categorical columns"
        )

        return self

    # =====================================================
    # Feature Scaling
    # =====================================================

    def normalize_features(self):
        """
        Standardize numerical features
        """

        print("🔍 Normalizing features...")

        numeric_cols = ['transaction_amount']

        scaler = StandardScaler()

        self.df[numeric_cols] = scaler.fit_transform(
            self.df[numeric_cols]
        )

        self.report['scaler'] = scaler

        print("✅ Features normalized")

        return self

    # =====================================================
    # Data Quality Report
    # =====================================================

    def generate_report(self):

        print("\n" + "=" * 60)
        print("📊 DATA QUALITY REPORT")
        print("=" * 60)

        print(f"Total Records: {len(self.df):,}")

        print(
            f"Duplicates Removed: "
            f"{self.report.get('duplicates_removed', 0)}"
        )

        missing_before = (
            self.report['missing_values_before'].sum()
            if 'missing_values_before' in self.report
            else 0
        )

        missing_after = (
            self.report['missing_values_after'].sum()
            if 'missing_values_after' in self.report
            else 0
        )

        print(
            f"Missing Values: "
            f"{missing_before} → {missing_after}"
        )

        print(
            f"Outliers Handled: "
            f"{self.report.get('outliers_handled', 0)}"
        )

        print(
            f"Fraud Cases: "
            f"{self.df['is_fraud'].sum():,}"
        )

        print(
            f"Fraud Percentage: "
            f"{self.df['is_fraud'].mean()*100:.2f}%"
        )

        print("=" * 60)

        return self.report

    # =====================================================
    # Return Clean Data
    # =====================================================

    def get_cleaned_data(self):
        """
        Return cleaned dataframe
        """
        return self.df

In [22]:
from src.data_cleaner import DataCleaner

cleaner = DataCleaner(df_raw)

cleaner.handle_missing_values() \
       .remove_duplicates() \
       .handle_outliers() \
       .generate_report()

df_clean = cleaner.get_cleaned_data()

Handling missing values...
Done


AttributeError: 'DataCleaner' object has no attribute 'remove_duplicates'